In [1]:
import os

from dotenv import load_dotenv

from openai import OpenAI

from openai.types.responses import ResponseTextDeltaEvent

from agents import Agent, Runner, function_tool, SQLiteSession, RunConfig

from agents.models.openai_provider import OpenAIProvider

import wikipedia

import webbrowser

import urllib.parse

import re
import urllib.parse
import urllib.request
import webbrowser

import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

load_dotenv(override=True)

True

In [3]:
provider = OpenAIProvider(
api_key=os.getenv("GROQ_API_KEY"),
base_url="https://api.groq.com/openai/v1",
use_responses=False
)

In [4]:
theconfig=RunConfig(model_provider=provider)

now lets make tools

In [5]:
@function_tool
def savetoafile(message:str,filename:str="response.txt"):
    """this is function that saves a text on a file """
    payload=message

    with open(filename,"w",encoding="utf=8")as f:
        f.write(payload)

    return f"message saved to the file {filename}"

@function_tool
def readfromafile(filename: str = "response.txt") -> str:
    """Reads and returns the content from a specified file."""
    try:
        with open(filename, "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        return f"Error: The file '{filename}' does not exist."
    except Exception as e:
        return f"Error reading file '{filename}': {str(e)}"

@function_tool

def wikisearch(query:str)->str:

    """Search Wikipedia and return a summary of the topic.

    Use this tool when you need factual information about a person, place, event, or concept. and if user ask to search for something etc..."""

    try:

        page = wikipedia.page(query, auto_suggest=True)

        return f"**{page.title}**\n\n{page.summary}\n\nSource: {page.url}"

    except wikipedia.DisambiguationError as e:

        # Multiple matches — pick the first option

        page = wikipedia.page(e.options[0])

        return f"**{page.title}**\n\n{page.summary}\n\nSource: {page.url}"

    except wikipedia.PageError:

        return f"No Wikipedia article found for '{query}'."

@function_tool

def exactdateandtimeoftoday(date:str)->str:

    """use this tool when you want exact information of the date and time of today before calling any other function call this function to stay up to date"""
    from datetime import datetime

    now = datetime.now()

    return now.strftime("%A, %B %d, %Y — %I:%M %p")


@function_tool

def turnthelightson(room:str)->str:

    """use this tool to turn the lights on of any room the room is given in the parameter"""

    print(f"light on of {room}")


@function_tool
def play_on_youtube(topic: str) -> str:
    """Finds and immediately plays a requested song or video on YouTube in the default browser.

    Args:
        topic: The name of the song, artist, video, or query to play.

    Returns:
        A confirmation message with the video URL opened.
    """
    encoded_query = urllib.parse.quote(topic)
    search_url = f"https://www.youtube.com/results?search_query={encoded_query}"

    try:
        # Request search results HTML
        req = urllib.request.Request(
            search_url,
            headers={"User-Agent": "Mozilla/5.0 (X11; Linux x86_64)"}
        )
        with urllib.request.urlopen(req, timeout=5) as response:
            html = response.read().decode("utf-8")

        # Extract the first valid 11-character video ID
        video_ids = re.findall(r"watch\?v=([a-zA-Z0-9_-]{11})", html)

        if video_ids:
            video_id = video_ids[0]
            # Build autoplay link with YouTube Mix/Radio parameter
            play_url = f"https://www.youtube.com/watch?v={video_id}&list=RD{video_id}&start_radio=1"
            webbrowser.open(play_url)
            return f"Now playing '{topic}' on YouTube: {play_url}"

    except Exception:
        # Fall back to opening search results if network or parsing fails
        pass

    webbrowser.open(search_url)
    return f"Opened YouTube search results for '{topic}'."

@function_tool

def email_sender(message:str,reciver:str,subject:str,bodyofmail:str)->str:
        """this function sends email to a recipent"""
        SMTP_SERVER = "smtp.gmail.com"  # e.g., smtp.gmail.com, smtp.office365.com
        SMTP_PORT = 587                 # Port 587 for STARTTLS

        SENDER_EMAIL = "sabyasachipanda410@gmail.com"
        SENDER_PASSWORD = "amhrfrqrrpykrbed"  # Use an App Password, NOT your regular account password
        RECIPIENT_EMAIL = reciver

        # --- Create Message ---
        msg = MIMEMultipart()
        msg["From"] = SENDER_EMAIL
        msg["To"] = RECIPIENT_EMAIL
        msg["Subject"] = subject

        body = bodyofmail
        msg.attach(MIMEText(body, "plain"))

        # --- Send Email ---
        try:
            with smtplib.SMTP(SMTP_SERVER, SMTP_PORT) as server:
                server.starttls()  # Upgrade connection to secure TLS
                server.login(SENDER_EMAIL, SENDER_PASSWORD)
                server.send_message(msg)
                print("Email sent successfully!")
        except Exception as e:
            print(f"Failed to send email: {e}")
        
        return f"message successfully sent to {reciver}"

In [6]:
doer = Agent(
name="doer",
model="openai/gpt-oss-20b",
instructions="""you are a multipurpose tool caller and helpful chatbot , call specific tools acording to the user message
and be precise

also keep in mind that the output what you will generate that will be speaked so generate the output acordingly , i am using groq's text to speach engine thogh    

1- savetoafile -- use this whenever you have to save something on the file 
2- wikisearch -- use this whenever you need any information from the internet and to seek knowledge and ways to find things and reserch 
3- exactdateandtimeoftoday - use this whenever you will need the exact date and time , call this tool before calling the wikisearch to get the latest info every time 
4- turnthelightson - use this to turn on the lights of a particular room provided by user
5- play_on_youtube - use this tool to play some songs requested on youtube and also podcast 
6- readfromafile - read from a file for longer memory or if you want to recall anything
7- email_sender -- use this tool to send email to anyone specified by user
""",
tools=[savetoafile, wikisearch,exactdateandtimeoftoday,turnthelightson,play_on_youtube,readfromafile,email_sender] # all the tools 
)

give it a memory

In [7]:
session=SQLiteSession("123123")

In [8]:
result= await Runner.run(
    doer,
    input("what you want to do"),
    run_config=theconfig,
    session=session
)
finaloutput=result.final_output
print(finaloutput)

Hello! How can I help you today?


OPENAI_API_KEY is not set, skipping trace export


lets do speak output

In [9]:
import os
from groq import Groq
from IPython.display import Audio, display

# Initialize Groq client
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# 1. Run your agent
result = await Runner.run(
    doer,
    input("what you want to do: "),
    run_config=theconfig,
    session=session
)

finaloutput = result.final_output
print("Agent:", finaloutput)

# 2. Pass agent's response text into Groq TTS
if finaloutput:
    speech_response = client.audio.speech.create(
        model="canopylabs/orpheus-v1-english",
        voice="autumn",
        response_format="wav",
        input=finaloutput,
    )

    # 3. Automatically play audio response
    display(Audio(data=speech_response.read(), autoplay=True))

Agent: I’m doing great—thanks for asking! How can I help you today?


OPENAI_API_KEY is not set, skipping trace export


In [10]:
import os
import subprocess
from groq import Groq
from IPython.display import Audio, display

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def speak(text: str):
    """Generates audio via Groq TTS and plays it automatically through your system speakers."""
    if not text:
        return
    
    # 1. Generate speech
    response = client.audio.speech.create(
        model="canopylabs/orpheus-v1-english",
        voice="autumn",
        response_format="wav",
        input=text,
    )
    
    # 2. Play directly through Linux sound system (paplay / aplay)
    process = subprocess.Popen(["paplay"], stdin=subprocess.PIPE)
    process.communicate(input=response.read())



# 1. Get response from agent
result = await Runner.run(
    doer,
    input("what you want to do: "),
    run_config=theconfig,
    session=session
)

finaloutput = result.final_output
print("Agent:", finaloutput)

# 2. Automatically play speech output through speakers
speak(finaloutput)


OPENAI_API_KEY is not set, skipping trace export


Agent: Glad to hear it! Let me know if there’s anything you’d like to do or chat about.


OPENAI_API_KEY is not set, skipping trace export


make a voice input output system

In [11]:
import os
import subprocess
from groq import Groq
from IPython.display import Audio, display

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def record_audio(filename="input.wav"):
    """Records audio from microphone. Press ENTER to start and ENTER to stop."""
    input("👉 Press ENTER to START recording...")
    print("🎙️ Listening... Speak into your mic!")
    
    # Start recording using Linux native `arecord`
    process = subprocess.Popen(["arecord", "-f", "cd", "-t", "wav", "-q", filename])
    
    input("⏹️ Press ENTER to STOP recording...")
    process.terminate()
    process.wait()
    print("✅ Recording finished.\n")

def listen_and_transcribe(filename="input.wav") -> str:
    """Transcribes the recorded audio file using Groq Whisper API."""
    record_audio(filename)
    
    with open(filename, "rb") as file:
        transcription = client.audio.transcriptions.create(
            file=(filename, file.read()),
            model="whisper-large-v3-turbo",
        )
    text = transcription.text.strip()
    print(f"🗣️ You said: \"{text}\"")
    return text

def speak(text: str):
    """Converts response text to speech using Groq TTS and plays through Linux speakers."""
    if not text:
        return
    
    print("🔊 Speaking response...")
    response = client.audio.speech.create(
        model="canopylabs/orpheus-v1-english",
        voice="autumn",
        response_format="wav",
        input=text,
    )
    
    # Play directly through Linux sound system (paplay)
    process = subprocess.Popen(["paplay"], stdin=subprocess.PIPE)
    process.communicate(input=response.read())


In [12]:
# 1. Listen to your voice and convert to text
voice_input = listen_and_transcribe()

if voice_input:
    # 2. Run the agent with your spoken query
    result = await Runner.run(
        doer,
        voice_input,
        run_config=theconfig,
        session=session
    )

    finaloutput = result.final_output
    print(f"\n🤖 Agent: {finaloutput}\n")

    # 3. Speak the agent's output automatically
    speak(finaloutput)

🎙️ Listening... Speak into your mic!
✅ Recording finished.

🗣️ You said: "Thank you."

🤖 Agent: You’re welcome! If you have any more questions or need help with anything—just let me know.

🔊 Speaking response...


OPENAI_API_KEY is not set, skipping trace export


in future we will add more agents and orchistrate agents for deferent tasks